# E2E Detection Inference

This notebook provides simplified inference functionality for PyNAS trained vessel detection models.

In [ ]:
"""Simplified inference script for PyNAS trained vessel detection model."""

import os
import sys
sys.path.append('..')
from pathlib import Path
from typing import Optional, Dict

import torch
import numpy as np
import matplotlib.pyplot as plt

from datasets.RawVessels.loader import RawVesselsDataModule
from datasets.RawClassifier.loader import ClassifierDataModule

In [ ]:
class Inference:
    """Simplified vessel detection inference class adapted for classification."""
    
    def __init__(self, model_path: str, device: Optional[str] = None) -> None:
        """
        Initialize the inference class.
        
        Args:
            model_path (str): Path to the TorchScript model file
            device (Optional[str]): Device to run inference on. If None, auto-detect
        """
        self.model_path = Path(model_path)
        assert self.model_path.exists(), f'Model file not found: {model_path}'
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if device is None else torch.device(device)
        print(f'Using device: {self.device}')
        
        self.model = torch.jit.load(self.model_path, map_location=self.device)
        self.model.eval()
        print(f'Model loaded from {self.model_path}')
    
    def predict(self, input_tensor: torch.Tensor, threshold: float = 0.85) -> np.ndarray:
        """
        Perform classification inference on input tensor.
        
        Args:
            input_tensor (torch.Tensor): Input tensor (B, C, H, W)
            threshold (float): Threshold for binary classification
            
        Returns:
            np.ndarray: Binary predictions (B,) - 1 for vessel, 0 for non-vessel
        """
        input_tensor = input_tensor.to(self.device)
        
        with torch.no_grad():
            predictions = self.model(input_tensor)
        
        # Handle different output formats
        if predictions.shape[1] == 2:  # Two-class output
            vessel_probs = predictions[:, 1]  # Probability of vessel class
        else:  # Single output
            vessel_probs = predictions.squeeze(1)
        
        # Apply sigmoid if needed (check if already normalized)
        if vessel_probs.min() < 0 or vessel_probs.max() > 1:
            vessel_probs = torch.sigmoid(vessel_probs)
        
        binary_predictions = (vessel_probs.cpu().numpy() > threshold).astype(np.uint8)
        return binary_predictions
    
    def calculate_metrics(self, predictions: np.ndarray, targets: np.ndarray) -> Dict[str, float]:
        """
        Calculate basic evaluation metrics for classification.
        
        Args:
            predictions (np.ndarray): Binary predictions (B,)
            targets (np.ndarray): Ground truth labels (B,)
            
        Returns:
            Dict[str, float]: Dictionary of metrics
        """
        assert predictions.shape == targets.shape, f'Shape mismatch: {predictions.shape} vs {targets.shape}'
        
        tp = np.sum((predictions == 1) & (targets == 1))
        fp = np.sum((predictions == 1) & (targets == 0))
        fn = np.sum((predictions == 0) & (targets == 1))
        tn = np.sum((predictions == 0) & (targets == 0))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        
        return {
            'precision': precision, 
            'recall': recall, 
            'f1_score': f1_score, 
            'accuracy': accuracy
        }
    
    def visualize(self, image: np.ndarray, prediction: int, target: int, idx: int = 0) -> None:
        """
        Visualize classification results.
        
        Args:
            image (np.ndarray): Input image (H, W) or (C, H, W)
            prediction (int): Prediction (0 or 1)
            target (int): Ground truth label (0 or 1)
            idx (int): Index for saving the image
        """
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        
        # Handle different image formats
        if len(image.shape) == 3:
            if image.shape[0] == 3:  # (C, H, W) format
                image = np.transpose(image, (1, 2, 0))  # Convert to (H, W, C)
            elif image.shape[2] == 1:  # (H, W, 1) format
                image = image.squeeze(2)  # Convert to (H, W)
        
        # Display image
        if len(image.shape) == 3:  # RGB image
            ax.imshow(image)
        else:  # Grayscale image
            ax.imshow(image, cmap='gray')
        
        # Color border based on correctness
        border_color = 'green' if prediction == target else 'red'
        ax.add_patch(plt.Rectangle((0, 0), image.shape[1]-1, image.shape[0]-1, 
                                 fill=False, edgecolor=border_color, linewidth=4))
        
        # Add text with prediction and target
        pred_text = 'Event' if prediction == 0 else 'No Event'
        target_text = 'Event' if target == 0 else 'No Event'
        status = '✓ Correct' if prediction == target else '✗ Incorrect'
        
        text = f'Prediction: {pred_text}\nGround Truth: {target_text}\n{status}'
        ax.text(0.02, 0.98, text, transform=ax.transAxes, fontsize=12, 
                verticalalignment='top', bbox=dict(boxstyle='round', 
                facecolor='white', alpha=0.8))
        
        ax.set_title(f'Classification Result - Sample {idx}', fontsize=14, fontweight='bold')
        ax.axis('off')
        
        plt.tight_layout()
        basedir = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/outputs'
        plt.savefig(f'{basedir}/classification_result_{idx}.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    def visualize_batch(self, images: torch.Tensor, predictions: np.ndarray, 
                       targets: np.ndarray, metrics: Optional[Dict[str, float]] = None,
                       max_samples: int = 8) -> None:
        """
        Visualize a batch of classification results.
        
        Args:
            images (torch.Tensor): Batch of images (B, C, H, W)
            predictions (np.ndarray): Batch predictions (B,)
            targets (np.ndarray): Batch targets (B,)
            metrics (Optional[Dict[str, float]]): Overall metrics
            max_samples (int): Maximum number of samples to display
        """
        batch_size = min(images.shape[0], max_samples)
        cols = min(4, batch_size)
        rows = (batch_size + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
        if batch_size == 1:
            axes = [axes]
        elif rows == 1:
            axes = axes.reshape(1, -1)
        
        for i in range(batch_size):
            row, col = i // cols, i % cols
            ax = axes[row, col] if rows > 1 else axes[col]
            
            # Convert image to numpy if needed and handle channels
            img = images[i].cpu().numpy()
            if img.shape[0] == 3:  # RGB - convert from (C, H, W) to (H, W, C)
                img = np.transpose(img, (1, 2, 0))
                ax.imshow(img)  # RGB image, no colormap
            elif img.shape[0] == 1:  # Grayscale - convert from (C, H, W) to (H, W)
                img = img.squeeze(0)
                ax.imshow(img, cmap='gray')  # Grayscale image with gray colormap
            else:  # Assume (H, W) format
                ax.imshow(img, cmap='gray')
            
            # Border color based on correctness
            border_color = 'green' if predictions[i] == targets[i] else 'red'
            ax.add_patch(plt.Rectangle((0, 0), img.shape[1]-1, img.shape[0]-1, 
                                     fill=False, edgecolor=border_color, linewidth=3))
            
            # Labels
            pred_label = 'V' if predictions[i] == 1 else 'N'
            target_label = 'V' if targets[i] == 1 else 'N'
            ax.set_title(f'P:{pred_label} T:{target_label}', fontsize=10)
            ax.axis('off')
        
        # Hide unused subplots
        for i in range(batch_size, rows * cols):
            row, col = i // cols, i % cols
            ax = axes[row, col] if rows > 1 else axes[col]
            ax.axis('off')
        
        # Add metrics text if provided
        if metrics is not None:
            metrics_text = 'Metrics:\n' + '\n'.join([f'{k.replace("_", " ").title()}: {v:.3f}' 
                                                    for k, v in metrics.items()])
            fig.text(0.02, 0.98, metrics_text, fontsize=10, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        
        plt.tight_layout()
        basedir = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/Outputs'
        plt.savefig(f'{basedir}/batch_classification_results.png', dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
# Configuration
model_path = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/Results_store/E2E/run1/models_traced/generation_8/model_and_architecture_5.pt'
data_dir = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/data/end2end'
batch_size = 1
threshold = 0.5

# Initialize inference
inferencer = Inference(model_path)
# Setup data module
data_module = ClassifierDataModule(root_dir=data_dir, batch_size=batch_size, mode='semisplit')
data_module.setup()

print(f'Data module setup complete:')
print(f'  Input shape: {data_module.input_shape}')
print(f'  Number of classes: {data_module.num_classes}')
print('=' * 50)
print(f'  Train dataset size: {len(data_module.train_dataset):,}')
print(f'  Validation dataset size: {len(data_module.val_dataset):,}')
print(f'  Test dataset size: {len(data_module.test_dataset):,}')
print('=' * 50)
print(f'  Batch size: {batch_size}')
print(f'  Data directory: {data_dir}')

In [ ]:
from IPython.display import clear_output

# Get a test sample and run inference
test_loader = data_module.test_dataloader()
# Get multiple test samples for comparison
test_samples = []
for i, (batch_images, batch_targets) in enumerate(test_loader):
    test_samples.append((batch_images, batch_targets))
    if i >= 500:  # Get 40 samples total
        break

# Select sample at index 2
for idx in range(len(test_samples)):
    images, targets = test_samples[idx]

    # Run inference
    predictions = inferencer.predict(images, threshold=0.85)

    # Extract data for visualization
    image = images[0].numpy()  # First image, first channel
    prediction = predictions[0]   # First prediction
    
    # Fix targets handling - targets is a tensor, not a tuple
    if len(targets.shape) == 2 and targets.shape[1] == 2:  # One-hot encoded
        target = targets[0, 1].numpy()  # Get the vessel class probability/label
    else:  # Single label
        target = targets[0].squeeze().numpy()

    # Calculate and display metrics
    metrics = inferencer.calculate_metrics(predictions, target.reshape(1, *target.shape))

    # Visualize results - use the correct method name
    image = image/image.max()
    # RGB to BGR
    image = image[::-1]  # Convert RGB to BGR by reversing channel order
    inferencer.visualize(image, prediction, target, idx=idx)
    print(f'Metrics: {metrics}')
    clear_output(wait=True)
    
    if idx >= 20:  # Limit to first 10 samples for brevity
        break


# # Get a test sample and run inference
# test_loader = data_module.test_dataloader()
# # Get multiple test samples for comparison
# test_samples = []
# for i, (batch_images, batch_targets) in enumerate(test_loader):
#     test_samples.append((batch_images, batch_targets))
#     if i >= 240:  # Get 40 samples total
#         break

# # Select sample at index 2
# idx = 4
# images, targets = test_samples[idx]

# # Run inference
# predictions = inferencer.predict(images, threshold=threshold)

# # Extract data for visualization
# image = images[0, 0].numpy()  # First image, first channel
# prediction = predictions[0]   # First prediction
# target = targets[0, 1].numpy() if targets.shape[1] == 2 else targets[0].squeeze().numpy()

# # Calculate and display metrics
# metrics = inferencer.calculate_metrics(predictions, target.reshape(1, *target.shape))

# # Visualize results
# inferencer.visualize2(image, prediction, target, pred_alpha=0.6, gt_alpha=0.6, metrics=metrics, idx=idx)
# print(f'Metrics: {metrics}')